In [ ]:
import astropy.units as u
from astropy.coordinates import SkyCoord

from astroquery.mast import MastMissions, Observations, Catalogs
from astropy.table import Table
from mast_aladin import MastTable, AppSidecar, MastAladin


def missions_mast(mission, env):
    missions = MastMissions(mission=mission)

    if env in ['dev', 'test', 'int']:
        base_url = f'https://mast{env}.stsci.edu'
        missions._service_api_connection.SERVICE_URL = base_url
        missions._service_api_connection.REQUEST_URL = f'{base_url}/search/{mission}/api/v0.1/'
        missions._service_api_connection.MISSIONS_DOWNLOAD_URL = f'{base_url}/search/'

    return missions

# Get obs data ready
mast = missions_mast('roman', 'test')
target = SkyCoord(ra=270, dec=66, unit='deg')
radius = 0.3 * u.deg
# obs = mast.query_region(target, radius=radius) #, select_cols=['s_region'])
obs = Table.read("roman_obs.csv")
obs_no_fp = Table(obs)
obs_no_fp.remove_column('s_region')

# Get catalog data ready
# gaia_sources = Catalogs.query_region(target, radius=0.2, catalog="Gaia", version=2)
gaia_sources = Table.read("gaia_sources.csv")

In [ ]:
aladin = MastAladin(
    target=f'{target.ra.deg} {target.dec.deg}',
    fov=0.5
)
apps_initialized = AppSidecar.open(aladin, anchor='split-right')
AppSidecar.resize_all(800)

All selection listening is handled in `MastTable`.  It listens for Aladin's `select` event and uses that payload to decide what to select in the `MastTable` instance.

In [ ]:
mast_table_obs_no_fp = MastTable(obs_no_fp, app=aladin)
mast_table_obs_no_fp

In [ ]:
obs_no_fp_layer = aladin.add_table(obs_no_fp, name='Roman Observations NO FPs', color='#FF6600', ra_field="ra_ref", dec_field="dec_ref")

In [ ]:
gaia_layer = aladin.add_table(gaia_sources)